# WaferGuard — PyTorch supervised autoencoder audit

**Decision: NO-GO for production based on the currently documented evidence.** This notebook uses a chronological train/validation/test split, training-only preprocessing, natural-prevalence evaluation, and an explicit validation-only threshold choice. Run all cells from a clean kernel to generate fresh metrics. No result is asserted before that run.

## Audit and repair notes

This notebook is designed to make the relevant safeguards executable and auditable:

1. **Leakage:** rows are ordered by `timestamp` before splitting; imputation and scaling are fitted on the training partition only. SMOTE is not used, so synthetic samples cannot cross partition boundaries and evaluation retains natural prevalence.
2. **Reconstruction-loss shape:** model inputs are consistently `[batch, features]`; reconstruction targets have the same shape. An assertion checks this before calculating MSE, preventing accidental broadcasting such as `[B, 1, F]` against `[B, F]`.
3. **Legacy PS-L encoding:** `sensor_0 == 0.0` is treated as a legacy sentinel, not a continuous measurement. `sensor_1` values 100/200/300 are represented as categorical indicators. The raw `sensor_0` and `sensor_1` columns are excluded from continuous imputation and scaling.
4. **Evaluation:** the threshold is selected using validation predictions only. Test metrics and bootstrap confidence intervals are calculated once from the held-out chronological test partition. They must be generated by running this notebook; no stale output is treated as evidence.

The dataset is expected at `data/raw/secom_telemetry_modified.csv` or as `secom_telemetry_modified.csv` beside the notebook.

## 0 · Environment

In [ ]:
%pip install "torch>=2.2" "scikit-learn>=1.2" pandas numpy joblib

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix
)
import joblib

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HP = dict(batch_size=64, epochs=40, lr=1e-3, weight_decay=1e-5,
          hidden=256, latent=32, lambda_clf=1.0,
          train_fraction=0.70, validation_fraction=0.15)
print(f"torch {torch.__version__} | device={DEVICE}")

## 1. Data


In [ ]:
candidate_paths = [
    os.path.join("data", "raw", "secom_telemetry_modified.csv"),
    "secom_telemetry_modified.csv",
]
data_path = next((p for p in candidate_paths if os.path.isfile(p)), None)
if data_path is None:
    raise FileNotFoundError(
        "Place secom_telemetry_modified.csv in data/raw/ or beside this notebook."
    )

df = pd.read_csv(data_path)
SENSOR_COLS = sorted(
    [c for c in df.columns if c.startswith("sensor_")],
    key=lambda c: int(c.split("_")[1])
)
required = {"label", "timestamp", "sensor_0", "sensor_1"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")
if not SENSOR_COLS:
    raise ValueError("No sensor_* columns were found.")

# Use chronological order to ensure later lots cannot influence earlier predictions.
df = df.copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
if df["timestamp"].isna().any():
    raise ValueError("timestamp contains missing or invalid values; resolve them before splitting.")
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)

y = (df["label"].to_numpy() == 1).astype(np.int64)
legacy_sentinel = df["sensor_0"].to_numpy(dtype=np.float64) == 0.0
sensor1 = df["sensor_1"].to_numpy(dtype=np.float64)
legacy_code_100 = sensor1 == 100.0
legacy_code_200 = sensor1 == 200.0
legacy_code_300 = sensor1 == 300.0

# Keep sentinel/code semantics outside the continuous imputer and scaler.
continuous_cols = [c for c in SENSOR_COLS if c not in {"sensor_0", "sensor_1"}]
X_cont_raw = df[continuous_cols].to_numpy(dtype=np.float64)
legacy_flags = np.column_stack([
    legacy_sentinel, legacy_code_100, legacy_code_200, legacy_code_300
]).astype(np.float64)
print(f"Rows: {len(df)} | continuous sensors: {len(continuous_cols)}")
print(f"Natural defect prevalence: {y.mean():.2%} ({int(y.sum())} defects / {int((1-y).sum())} non-defects)")
print(f"Missing continuous cells: {np.isnan(X_cont_raw).mean():.2%}")
print(f"sensor_0 == 0 sentinel rows: {int(legacy_sentinel.sum())}")
print(f"sensor_1 categorical code counts: 100={int(legacy_code_100.sum())}, "
      f"200={int(legacy_code_200.sum())}, 300={int(legacy_code_300.sum())}")

## 2 · Preprocessing & class rebalancing


In [ ]:
# Chronological split: earliest rows train, following rows validate, latest rows test.
n = len(df)
n_train = int(n * HP["train_fraction"])
n_val = int(n * HP["validation_fraction"])
i_train = np.arange(0, n_train)
i_val = np.arange(n_train, n_train + n_val)
i_test = np.arange(n_train + n_val, n)

if min(len(i_train), len(i_val), len(i_test)) == 0:
    raise ValueError("Dataset is too small for the configured chronological split.")

X_train_raw, X_val_raw, X_test_raw = X_cont_raw[i_train], X_cont_raw[i_val], X_cont_raw[i_test]
F_train, F_val, F_test = legacy_flags[i_train], legacy_flags[i_val], legacy_flags[i_test]
y_train, y_val, y_test = y[i_train], y[i_val], y[i_test]

# Fit both transformations on training data only.
imputer = SimpleImputer(strategy="median", keep_empty_features=True)
X_train_imp = imputer.fit_transform(X_train_raw)
X_val_imp = imputer.transform(X_val_raw)
X_test_imp = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train_cont = scaler.fit_transform(X_train_imp)
X_val_cont = scaler.transform(X_val_imp)
X_test_cont = scaler.transform(X_test_imp)

# Append unscaled semantic indicators; do not impute/scalе the sentinel/categorical columns.
X_train = np.concatenate([X_train_cont, F_train], axis=1).astype(np.float32)
X_val = np.concatenate([X_val_cont, F_val], axis=1).astype(np.float32)
X_test = np.concatenate([X_test_cont, F_test], axis=1).astype(np.float32)

print(f"chronological split: train={X_train.shape}, validation={X_val.shape}, test={X_test.shape}")
print(f"defect prevalence: train={y_train.mean():.2%}, validation={y_val.mean():.2%}, test={y_test.mean():.2%}")
print(f"split boundaries: train ends {df.loc[i_train[-1], 'timestamp']}, "
      f"validation ends {df.loc[i_val[-1], 'timestamp']}, test ends {df.loc[i_test[-1], 'timestamp']}")

## 3 · Tensors & loaders


In [ ]:
F = X_train.shape[1]

def make_loader(X, y, shuffle):
    Xt = torch.from_numpy(np.asarray(X, dtype=np.float32))  # [N, F]
    yt = torch.from_numpy(np.asarray(y, dtype=np.float32))  # [N]
    assert Xt.ndim == 2 and Xt.shape[0] == yt.shape[0]
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(
        TensorDataset(Xt, yt), batch_size=HP["batch_size"],
        shuffle=shuffle, generator=generator, drop_last=False
    )

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)
print(f"features={F} | train batches={len(train_loader)} | validation batches={len(val_loader)} | test batches={len(test_loader)}")

## 4 · Model — supervised autoencoder


In [ ]:
class WaferSAE(nn.Module):
    def __init__(self, d_in: int, d_hid: int, d_lat: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in),
        )
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x):
        if x.ndim != 2:
            raise ValueError(f"Expected [batch, features], got {tuple(x.shape)}")
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        logits = self.head(z).squeeze(-1)
        assert reconstruction.shape == x.shape, (reconstruction.shape, x.shape)
        assert logits.shape == (x.shape[0],), (logits.shape, x.shape)
        return reconstruction, logits, z

model = WaferSAE(F, HP["hidden"], HP["latent"]).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5 · Training


In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"])
# Class weighting uses training labels only; no resampling changes natural prevalence.
n_pos = max(int(y_train.sum()), 1)
n_neg = max(int((1 - y_train).sum()), 1)
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=DEVICE)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
history = {"total": [], "recon": [], "clf": []}

for epoch in range(1, HP["epochs"] + 1):
    model.train()
    totals = dict(total=0.0, recon=0.0, clf=0.0)
    n_seen = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        reconstructed, logits, _ = model(batch_x)
        assert reconstructed.shape == batch_x.shape
        assert logits.shape == batch_y.shape
        recon_loss = torch.mean((reconstructed - batch_x) ** 2)
        clf_loss = bce(logits, batch_y)
        loss = recon_loss + HP["lambda_clf"] * clf_loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        bs = batch_x.shape[0]
        totals["total"] += loss.item() * bs
        totals["recon"] += recon_loss.item() * bs
        totals["clf"] += clf_loss.item() * bs
        n_seen += bs
    sched.step()
    for k in totals:
        history[k].append(totals[k] / n_seen)
    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:02d}/{HP['epochs']} | total {history['total'][-1]:.4f} "
              f"| recon {history['recon'][-1]:.4f} | clf {history['clf'][-1]:.4f}")

## 6 · Evaluation


In [ ]:
def predict_probabilities(loader):
    model.eval()
    chunks = []
    with torch.no_grad():
        for xb, _ in loader:
            _, logits, _ = model(xb.to(DEVICE))
            chunks.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(chunks)

val_probs = predict_probabilities(val_loader)
test_probs = predict_probabilities(test_loader)

# Choose threshold on validation data only; test labels do not influence this choice.
threshold_grid = np.unique(np.concatenate([
    np.linspace(0.01, 0.99, 99), val_probs
]))
val_f1 = np.array([f1_score(y_val, val_probs >= t, zero_division=0) for t in threshold_grid])
best_idx = int(np.argmax(val_f1))
threshold = float(threshold_grid[best_idx])
preds = (test_probs >= threshold).astype(int)

f1 = f1_score(y_test, preds, zero_division=0)
prec = precision_score(y_test, preds, zero_division=0)
rec = recall_score(y_test, preds, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()
auc = roc_auc_score(y_test, test_probs) if len(np.unique(y_test)) == 2 else float("nan")
ap = average_precision_score(y_test, test_probs) if y_test.sum() else float("nan")

# Percentile bootstrap CIs resample held-out test rows. These describe sampling uncertainty,
# not uncertainty from retraining or changing the temporal split.
rng = np.random.default_rng(SEED)
auc_boot, ap_boot = [], []
if len(np.unique(y_test)) == 2 and y_test.sum() >= 2 and (1-y_test).sum() >= 2:
    for _ in range(2000):
        idx = rng.integers(0, len(y_test), size=len(y_test))
        if len(np.unique(y_test[idx])) < 2:
            continue
        auc_boot.append(roc_auc_score(y_test[idx], test_probs[idx]))
        ap_boot.append(average_precision_score(y_test[idx], test_probs[idx]))

def ci95(values):
    return tuple(np.percentile(values, [2.5, 97.5])) if len(values) else (float("nan"), float("nan"))

auc_ci = ci95(auc_boot)
ap_ci = ci95(ap_boot)
print(f"Validation-selected threshold: {threshold:.4f} | validation F1={val_f1[best_idx]:.4f}")
print(f"Test prevalence: {y_test.mean():.2%} ({int(y_test.sum())}/{len(y_test)})")
print(f"F1={f1:.4f} | precision={prec:.4f} | recall={rec:.4f}")
print(f"ROC-AUC={auc:.4f} | bootstrap 95% CI [{auc_ci[0]:.4f}, {auc_ci[1]:.4f}]")
print(f"Average precision={ap:.4f} | bootstrap 95% CI [{ap_ci[0]:.4f}, {ap_ci[1]:.4f}]")
print(f"Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

## 7 · Release decision

The model is **NO-GO for production** until the fresh chronological evaluation demonstrates useful defect-ranking performance and the operational team validates the false-alarm/miss trade-off. The test partition is not used to select the threshold. Metrics and confidence intervals printed above are generated at the test partition's natural prevalence.

In [ ]:
GATE_F1 = 0.90
verdict = "NOT MET" if f1 < GATE_F1 else "MET — requires operational review"
print(f"Reference F1 gate (≥ {GATE_F1:.2f}): {f1:.4f} → {verdict}")
print("Release decision: NO-GO pending evidence of useful performance and operational validation.")

os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/wafer_sae.pt")
joblib.dump(
    {"imputer": imputer, "scaler": scaler,
     "continuous_cols": continuous_cols,
     "feature_names": continuous_cols + [
         "legacy_sensor0_zero_sentinel", "sensor1_code_100",
         "sensor1_code_200", "sensor1_code_300"
     ],
     "threshold": threshold},
    "artifacts/preprocessor.joblib"
)
print("Saved model weights and fitted training-only preprocessing metadata to ./artifacts/.")